##**Build and Evaluate a Sentiment Classifier (Naive Bayes + Feature Engineering)**

In [1]:
import nltk
from nltk.corpus import movie_reviews
import random

nltk.download('movie_reviews')
docs = [(list(movie_reviews.words(fileid)),category)
        for category in movie_reviews.categories()
        for fileid in movie_reviews.fileids(category)]

random.shuffle(docs)

def extract_feature(words):
  return {word: True for word in words}

featuresets = [(extract_feature(doc),category) for (doc,category) in docs]
train_set,test_set = featuresets[:1500],featuresets[1500:]

classifier = nltk.NaiveBayesClassifier.train(train_set)
print("Accuracy:", nltk.classify.accuracy(classifier, test_set))
classifier.show_most_informative_features(10)

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


Accuracy: 0.698
Most Informative Features
               insulting = True              neg : pos    =     15.1 : 1.0
                   sucks = True              neg : pos    =     13.7 : 1.0
                  seagal = True              neg : pos    =     12.3 : 1.0
                attorney = True              pos : neg    =     11.7 : 1.0
                  avoids = True              pos : neg    =     11.1 : 1.0
                gripping = True              pos : neg    =     10.4 : 1.0
                poignant = True              pos : neg    =     10.1 : 1.0
               animators = True              pos : neg    =      9.8 : 1.0
                 tribute = True              pos : neg    =      9.8 : 1.0
                  alicia = True              neg : pos    =      9.5 : 1.0


# **N-gram Phrase Extraction with Frequency Filtering**

In [2]:
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

words = movie_reviews.words()
finder = BigramCollocationFinder.from_words(words)
finder.apply_freq_filter(20)

print(finder.nbest(BigramAssocMeasures.pmi,10))  # Top 10 meaningful bigrams

[('del', 'toro'), ('salma', 'hayek'), ('san', 'francisco'), ('mortal', 'kombat'), ('charlize', 'theron'), ('ace', 'ventura'), ('natalie', 'portman'), ('ewan', 'mcgregor'), ('los', 'angeles'), ('darth', 'vader')]


# **Word2Vec-based Similarity Search using NLTK Corpus**

In [3]:
import nltk
nltk.download('brown')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 38.7 MB/s eta 0:00:00


In [5]:
import gensim
from nltk.corpus import brown

sentences = brown.sents()
model = gensim.models.Word2Vec(sentences,vector_size=100,window=5,min_count=5,sg=1)

print("Similar to 'money':",model.wv.most_similar('money'))

Similar to 'money': [('job', 0.8294438123703003), ('care', 0.8209854960441589), ('friendship', 0.8151518702507019), ('joy', 0.8094804286956787), ('bringing', 0.8013061881065369), ('anywhere', 0.7975176572799683), ('comfort', 0.7961663603782654), ('luck', 0.795605480670929), ('permission', 0.7945327162742615), ('part-time', 0.7935254573822021)]


# **Doc2Vec for Document Classification**

In [6]:
from gensim.models.doc2vec import TaggedDocument,Doc2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

docs = [TaggedDocument(words=movie_reviews.words(fileid),tags=[fileid])
        for fileid in movie_reviews.fileids()]

model = Doc2Vec(docs,vector_size = 50,epochs=100)

X = [model.dv[doc.tags[0]] for doc in docs]
y = [movie_reviews.categories(fileid)[0] for fileid in movie_reviews.fileids()]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

clf = LogisticRegression()
clf.fit(X_train,y_train)
print("Doc2Vec Classifier Accuracy:", accuracy_score(y_test, clf.predict(X_test)))

Doc2Vec Classifier Accuracy: 0.75


# **PPMI Co-occurrence Vector Construction**

In [7]:
from collections import Counter
import numpy as np
from nltk.util import bigrams

text = "dog cat bark meow dog bark meow cat".split()
vocab = list(set(text))
co_matrix = Counter(bigrams(text))

word_counts = Counter(text)
ppmi_matrix = {}

for (w1,w2),count in co_matrix.items():
  p_w1 = word_counts[w1]/len(text)
  p_w2 = word_counts[w2]/len(text)
  p_w1_w2 = count/len(text)
  pmi = np.log2(p_w1_w2 / (p_w1 * p_w2))
  ppmi_matrix[(w1,w2)] = max(pmi,0)
print("PPMI Matrix:",ppmi_matrix)

PPMI Matrix: {('dog', 'cat'): np.float64(1.0), ('cat', 'bark'): np.float64(1.0), ('bark', 'meow'): np.float64(2.0), ('meow', 'dog'): np.float64(1.0), ('dog', 'bark'): np.float64(1.0), ('meow', 'cat'): np.float64(1.0)}


# **NER Visualization with Displacy for Legal Contracts**

In [9]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [10]:
from spacy import displacy

doc = nlp("This Agreement is made on 4th April 2023 between Apple Inc. and John Doe.")
displacy.render(doc, style="ent", jupyter=True)

# **Custom Pattern Matching (e.g., Date + Entity)**

In [11]:
from spacy.matcher import Matcher
matcher = Matcher(nlp.vocab)

pattern = [{"ENT_TYPE":"DATE"},{"LOWER":"between"},{"ENT_TYPE": "ORG"}]
matcher.add("DATE_CONTRACT_PATTERN",[pattern])

doc = nlp("Signed on 5th March 2024 between Microsoft and John.")
matches = matcher(doc)

for match_id,start,end in matches:
  print(doc[start:end].text)

2024 between Microsoft


# **Text Clustering using spaCy Vectors**

In [12]:
from sklearn.cluster import KMeans
import numpy as np

sentences = ["The cat sits on the mat", "Dogs bark loudly", "The dog is in the yard"]
docs = [nlp(sent) for sent in sentences]
X = np.array([doc.vector for doc in docs])

kmeans = KMeans(n_clusters=2)
kmeans.fit(X)

for i,label in enumerate(kmeans.labels_):
  print(f"Cluster {label}:{sentences[i]}")

Cluster 0:The cat sits on the mat
Cluster 1:Dogs bark loudly
Cluster 0:The dog is in the yard


# **Fine-Grained NER Training (Custom Labels)**

In [13]:
import spacy
from spacy.training.example import Example
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# Start from a blank English model
nlp = spacy.blank("en")

# Add NER pipeline component
ner = nlp.add_pipe("ner")
ner.add_label("ORG")
ner.add_label("PRODUCT")

# Create a sample training set with diverse examples
TRAIN_DATA = [
    ("Apple released the new Vision Pro.", {"entities": [(0, 5, "ORG"), (24, 34, "PRODUCT")]}),
    ("Microsoft launched Surface Laptop.", {"entities": [(0, 9, "ORG"), (19, 34, "PRODUCT")]}),
    ("Google unveiled the Pixel 8 phone.", {"entities": [(0, 6, "ORG"), (19, 29, "PRODUCT")]}),
    ("Apple Vision Pro is a mixed reality headset.", {"entities": [(0, 5, "ORG"), (6, 16, "PRODUCT")]}),
    ("Amazon presented the new Echo Show 10.", {"entities": [(0, 6, "ORG"), (25, 39, "PRODUCT")]}),
]

# Convert to Example objects
examples = []
for text,annots in TRAIN_DATA:
  doc = nlp.make_doc(text)
  examples.append(Example.from_dict(doc,annots))

# Training loop
optimizer = nlp.begin_training()
for i in range(30):
  nlp.update(examples,sgd=optimizer)

# Test
test_doc = nlp("Apple introduced the Vision Pro headset today.")
print([(ent.text, ent.label_) for ent in test_doc.ents])

[('Apple', 'ORG'), ('Vision Pro', 'PRODUCT')]
